# Group Preprocessing Notebook

## Project Purpose and CRISP-DM Stage

This notebook prepares the shared transaction dataset for the group fraud detection modelling work. It supports the data understanding and data preparation stages of CRISP-DM.

The business aim is to detect high-risk fraudulent transactions while considering both missed fraud and incorrectly blocked genuine transactions.

## Import Libraries

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Project Paths

In [ ]:
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_PATH = DATA_RAW_DIR / "transactions.csv"
TARGET_COLUMN = "is_fraud"
RANDOM_STATE = 42

## Load Raw Dataset

Add the raw transaction dataset to `data/raw/` before running this notebook. Update `RAW_DATA_PATH` if the file name is different.

In [ ]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError("The raw dataset must be added to data/raw/ before running this notebook.")

df = pd.read_csv(RAW_DATA_PATH)
df.head()

## Inspect Dataset Shape, Columns, Data Types, Missing Values, and Duplicate Records

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

display(df.head())
display(df.dtypes.to_frame("data_type"))
display(df.isna().sum().to_frame("missing_values"))

duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count:,}")

## Identify Target Variable and Class Balance

In [ ]:
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Expected target column '{TARGET_COLUMN}' was not found in the dataset.")

class_balance = df[TARGET_COLUMN].value_counts(dropna=False).to_frame("count")
class_balance["percentage"] = df[TARGET_COLUMN].value_counts(normalize=True, dropna=False) * 100
display(class_balance)

## Data Cleaning Decisions

Record agreed cleaning decisions here before applying them. Keep each decision linked to evidence from the inspection above.

In [ ]:
# Example structure for cleaning once decisions are agreed:
# df_clean = df.copy()
# df_clean = df_clean.drop_duplicates()

df_clean = df.copy()

## Feature Engineering Ideas

Record candidate features that may improve fraud risk modelling, such as transaction timing, amount bands, customer behaviour patterns, or merchant-level signals if available.

In [ ]:
# Add feature engineering steps here once the final dataset columns are confirmed.
df_features = df_clean.copy()

## Encoding and Scaling Strategy

Document which variables need encoding and which numeric variables should be scaled for models such as Logistic Regression.

In [ ]:
# Select and transform features after confirming the final schema.
# Use one-hot encoding for nominal categories when appropriate.
# Scale numeric features after the train/validation/test split to avoid leakage.

model_df = df_features.copy()

## Train / Validation / Test Split

Use training data for fitting, validation data for model selection, and test data only for final evaluation.

In [ ]:
X = model_df.drop(columns=[TARGET_COLUMN])
y = model_df[TARGET_COLUMN]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

print(X_train.shape, X_val.shape, X_test.shape)

## Save Processed Datasets

In [ ]:
train_processed = X_train.copy()
train_processed[TARGET_COLUMN] = y_train

val_processed = X_val.copy()
val_processed[TARGET_COLUMN] = y_val

test_processed = X_test.copy()
test_processed[TARGET_COLUMN] = y_test

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train_processed.to_csv(DATA_PROCESSED_DIR / "train_processed.csv", index=False)
val_processed.to_csv(DATA_PROCESSED_DIR / "val_processed.csv", index=False)
test_processed.to_csv(DATA_PROCESSED_DIR / "test_processed.csv", index=False)

print("Processed datasets saved to data/processed/.")

## Summary Table of Final Selected Features

In [ ]:
feature_summary = pd.DataFrame({
    "feature_name": X_train.columns,
    "data_type": X_train.dtypes.astype(str).values,
    "included_in_model": True,
    "notes": "",
})

display(feature_summary)